# 🔢 Notebook 2: NumPy — Arrays, Matrices y Álgebra Lineal para ML
### Curso: Machine Learning Avanzado — Módulo 1

**Objetivo:** Comprender cómo NumPy implementa operaciones vectorizadas, manejo eficiente de memoria y álgebra lineal, base matemática de todos los algoritmos de ML.

---
## 🗺️ Contenido
1. Anatomía de un ndarray (dtype, shape, strides, buffer)
2. Creación eficiente de arrays
3. Indexing avanzado: fancy indexing, boolean masking
4. Broadcasting: el superpoder de NumPy
5. Álgebra Lineal: la base matemática de ML
6. Caso aplicado: Implementación desde cero de Regresión Lineal

---
## 1. Anatomía de un ndarray: Entendiendo la Estructura Interna

### 📖 Concepto
Un `ndarray` de NumPy NO es una lista de Python. Es un **bloque contiguo de memoria** con metadatos:

```
ndarray
  ├── data buffer    → bytes contiguos en RAM (C o Fortran order)
  ├── dtype          → tipo de dato (float32, int64, bool...)
  ├── shape          → dimensiones (filas, columnas, ...)
  ├── strides        → bytes a saltar para ir al siguiente elemento
  └── flags          → C_CONTIGUOUS, F_CONTIGUOUS, WRITEABLE...
```

**¿Por qué importa en ML?**
- `float32` usa la **mitad de memoria** que `float64` → modelos 2x más grandes en GPU
- `strides` explica por qué `.T` (transpuesta) es O(1): no copia datos
- Contiguidad determina si BLAS/LAPACK pueden usar vectorización SIMD

In [ ]:
import numpy as np
import sys

# ── Anatomía del ndarray ──────────────────────────────────────────────────
# Creamos matrices de diferentes tipos para comparar
A_f64 = np.random.randn(1000, 1000)              # float64 (default)
A_f32 = A_f64.astype(np.float32)                 # float32
A_f16 = A_f64.astype(np.float16)                 # float16 (mixed precision)

print("Comparación de dtypes — Matriz 1000x1000:")
print(f"  float64: {A_f64.nbytes / 1024**2:.2f} MB | dtype={A_f64.dtype} | strides={A_f64.strides}")
print(f"  float32: {A_f32.nbytes / 1024**2:.2f} MB | dtype={A_f32.dtype} | strides={A_f32.strides}")
print(f"  float16: {A_f16.nbytes / 1024**2:.2f} MB | dtype={A_f16.dtype} | strides={A_f16.strides}")

# Strides — cómo navega NumPy en memoria
B = np.array([[1, 2, 3], [4, 5, 6]], dtype=np.float64)
print(f"\nMatrix B shape={B.shape}, strides={B.strides}")
print("  → Mover 1 columna = 8 bytes (1 float64)")
print("  → Mover 1 fila    = 24 bytes (3 float64)")

# Transpuesta: NO copia datos, solo cambia strides
BT = B.T
print(f"\nB.T  shape={BT.shape}, strides={BT.strides}")
print(f"Comparten memoria: {np.shares_memory(B, BT)}")

# Vista vs Copia — crítico para evitar bugs en pipelines
B_view = B[0:2, 0:2]   # vista (comparte buffer)
B_copy = B[0:2, 0:2].copy()  # copia independiente
print(f"\nB_view comparte memoria con B: {np.shares_memory(B, B_view)}")
print(f"B_copy comparte memoria con B: {np.shares_memory(B, B_copy)}")
print("⚠️  Modificar B_view modifica B — ¡cuidado en pipelines!")

In [ ]:
# ── Creación eficiente de arrays para ML ──────────────────────────────────
np.random.seed(42)

# Inicializadores comunes en Deep Learning
n_inputs, n_outputs = 784, 256  # capas de red neuronal (MNIST)

# Xavier/Glorot Initialization
limit = np.sqrt(6 / (n_inputs + n_outputs))
W_xavier = np.random.uniform(-limit, limit, (n_inputs, n_outputs))

# He Initialization (para ReLU)
W_he = np.random.randn(n_inputs, n_outputs) * np.sqrt(2 / n_inputs)

print("Inicialización de pesos para red neuronal:")
print(f"  Xavier: mean={W_xavier.mean():.6f}, std={W_xavier.std():.6f}")
print(f"  He:     mean={W_he.mean():.6f}, std={W_he.std():.6f}")

# Creación de datasets sintéticos
# Clasificación binaria — dos nubes de puntos
n_samples = 500
X_clase0 = np.random.randn(n_samples, 2) + np.array([2, 2])
X_clase1 = np.random.randn(n_samples, 2) + np.array([-2, -2])
X_dataset = np.vstack([X_clase0, X_clase1])
y_dataset = np.hstack([np.zeros(n_samples), np.ones(n_samples)])

print(f"\nDataset sintético: X={X_dataset.shape}, y={y_dataset.shape}")
print(f"  Memoria X: {X_dataset.nbytes / 1024:.1f} KB")

# Shuffle reproducible
idx = np.random.permutation(len(y_dataset))
X_shuffled = X_dataset[idx]
y_shuffled = y_dataset[idx]
print(f"  Distribución de clases: {np.unique(y_shuffled, return_counts=True)[1]}")

In [ ]:
# ── Indexing Avanzado: Fancy Indexing y Boolean Masking ───────────────────

# Dataset: señales de sensores IoT industriales (temperatura)
np.random.seed(42)
n_sensores, n_lecturas = 10, 1000
temperaturas = np.random.normal(loc=70, scale=15, size=(n_sensores, n_lecturas))

# Boolean Masking: detectar lecturas anómalas (>3 desviaciones estándar)
media_global = temperaturas.mean()
std_global   = temperaturas.std()
anomalias_mask = np.abs(temperaturas - media_global) > 3 * std_global

print(f"Detección de anomalías en sensores industriales:")
print(f"  Total lecturas: {temperaturas.size:,}")
print(f"  Anomalías detectadas: {anomalias_mask.sum()} ({anomalias_mask.mean():.2%})")

# Qué sensores tienen más anomalías
anomalias_por_sensor = anomalias_mask.sum(axis=1)
sensor_critico = np.argmax(anomalias_por_sensor)
print(f"  Sensor más problemático: Sensor #{sensor_critico} "
      f"({anomalias_por_sensor[sensor_critico]} anomalías)")

# Fancy Indexing: seleccionar top-3 sensores con más anomalías
top_sensores = np.argsort(anomalias_por_sensor)[::-1][:3]
print(f"  Top 3 sensores críticos: {top_sensores}")
print(f"  Sus anomalías: {anomalias_por_sensor[top_sensores]}")

# Imputación de anomalías con la mediana del sensor
temperaturas_clean = temperaturas.copy()
medianas_sensores = np.median(temperaturas, axis=1, keepdims=True)
temperaturas_clean[anomalias_mask] = np.broadcast_to(
    medianas_sensores, temperaturas.shape
)[anomalias_mask]
print(f"\n  Temperatura media antes: {temperaturas.mean():.2f}°C")
print(f"  Temperatura media después: {temperaturas_clean.mean():.2f}°C")

---
## 2. Broadcasting: El Superpoder de NumPy

### 📖 Concepto
**Broadcasting** es el mecanismo por el que NumPy aplica operaciones entre arrays de **diferentes formas** sin copiar datos.

**Reglas de Broadcasting:**
1. Si los arrays tienen diferente número de dimensiones, se expande el de menor rango por la izquierda
2. Las dimensiones son compatibles si son iguales o si una de ellas es 1
3. El resultado tiene el máximo de cada dimensión

```
A: (1000, 20)   + B: (20,)   → (1000, 20)   ✅ Broadcasting
A: (1000, 20)   + B: (1000,) → ERROR         ❌ (necesita (1000, 1))
A: (1000, 20)   + B: (1000, 1) → (1000, 20) ✅ Broadcasting
```

In [ ]:
import timeit

# ── Broadcasting en práctica: Estandarización Z-Score ────────────────────
np.random.seed(42)
X = np.random.randn(10_000, 50).astype(np.float32)  # 10K muestras, 50 features

# SIN broadcasting (loop — malo)
def zscore_loop(X):
    X_norm = np.zeros_like(X)
    for j in range(X.shape[1]):
        X_norm[:, j] = (X[:, j] - X[:, j].mean()) / (X[:, j].std() + 1e-8)
    return X_norm

# CON broadcasting (vectorizado — óptimo)
def zscore_broadcast(X):
    mu = X.mean(axis=0)       # shape: (50,)
    sigma = X.std(axis=0)     # shape: (50,)
    return (X - mu) / (sigma + 1e-8)  # broadcasting: (10000,50)-(50,)

t_loop  = timeit.timeit(lambda: zscore_loop(X), number=100)
t_broad = timeit.timeit(lambda: zscore_broadcast(X), number=100)

print("Estandarización Z-Score (100 iteraciones, 10K × 50):")
print(f"  Loop columna a columna: {t_loop:.4f}s")
print(f"  Broadcasting vectorizado: {t_broad:.4f}s  ← {t_loop/t_broad:.1f}x más rápido")

X_norm = zscore_broadcast(X)
print(f"\nVerificación — media por feature (debe ser ≈0): max|μ|={np.abs(X_norm.mean(axis=0)).max():.2e}")
print(f"Verificación — std por feature  (debe ser ≈1): max|σ-1|={np.abs(X_norm.std(axis=0) - 1).max():.2e}")

---
## 3. Álgebra Lineal con NumPy: Base Matemática de ML

### 📖 Conceptos Fundamentales

| Operación | NumPy | Uso en ML |
|-----------|-------|-----------|
| Producto matricial | `A @ B` o `np.matmul` | Forward pass en redes neuronales |
| Transpuesta | `A.T` | Gradientes, correlación |
| Inversa | `np.linalg.inv` | Regresión lineal (ecuación normal) |
| Pseudo-inversa | `np.linalg.pinv` | Sistemas sobredeterminados |
| SVD | `np.linalg.svd` | PCA, compresión, recomendación |
| Eigenvalores | `np.linalg.eig` | PCA, estabilidad de sistemas |
| Normas | `np.linalg.norm` | Regularización L1/L2 |

In [ ]:
# ── Regresión Lineal: Implementación desde cero ───────────────────────────
# Caso Real: Predicción de precio de viviendas

np.random.seed(42)
n = 200  # muestras

# Features: [metros_cuadrados, n_habitaciones, antiguedad_años, distancia_centro_km]
X_raw = np.column_stack([
    np.random.uniform(40, 250, n),   # metros cuadrados
    np.random.randint(1, 6, n),      # habitaciones
    np.random.uniform(0, 50, n),     # antigüedad
    np.random.uniform(1, 30, n),     # distancia al centro
])

# Target: precio real (USD) con algo de ruido
true_coefs = np.array([1500, 25000, -800, -3000])
y_raw = X_raw @ true_coefs + 50000 + np.random.randn(n) * 15000

# --- MÉTODO 1: Ecuación Normal (solución exacta) -------------------------
# β = (XᵀX)⁻¹Xᵀy
X_bias = np.column_stack([np.ones(n), X_raw])  # agregar columna de bias

# Solución con ecuación normal
betas_normal = np.linalg.inv(X_bias.T @ X_bias) @ (X_bias.T @ y_raw)

# --- MÉTODO 2: Mínimos Cuadrados (más estable numéricamente) -------------
betas_lstsq, residuals, rank, sv = np.linalg.lstsq(X_bias, y_raw, rcond=None)

# Predicciones y métricas
y_pred = X_bias @ betas_lstsq
ss_res = np.sum((y_raw - y_pred) ** 2)
ss_tot = np.sum((y_raw - y_raw.mean()) ** 2)
r2 = 1 - ss_res / ss_tot
rmse = np.sqrt(ss_res / n)

features = ["bias", "m²", "habitaciones", "antigüedad", "distancia"]
print("Regresión Lineal — Predicción de Precio de Vivienda")
print("\nCoeficientes estimados vs reales:")
print(f"  {'Feature':15} | {'Estimado':>12} | {'Real':>12} | {'Error%':>8}")
print("  " + "-" * 55)
for fname, est, real in zip(features[1:], betas_lstsq[1:], true_coefs):
    err = abs(est - real) / abs(real) * 100
    print(f"  {fname:15} | {est:12,.0f} | {real:12,.0f} | {err:8.1f}%")

print(f"\nMétricas del modelo:")
print(f"  R² = {r2:.4f}")
print(f"  RMSE = ${rmse:,.0f}")

In [ ]:
# ── SVD: Fundamento de PCA y Sistemas de Recomendación ───────────────────
# Caso Real: Compresión de imagen (análogo a reducción dimensional)

# Simulamos una matrix de ratings de usuarios-películas (tipo Netflix)
np.random.seed(42)
n_users, n_movies = 100, 50

# Matriz latente: 5 factores reales (géneros: acción, drama, comedia, terror, sci-fi)
n_factores_reales = 5
U_real = np.random.randn(n_users, n_factores_reales)
V_real = np.random.randn(n_factores_reales, n_movies)
R = np.clip(U_real @ V_real + 3, 1, 5)  # ratings entre 1 y 5

# Agregar ruido y sparsity (75% de ratings faltantes)
R_noisy = R + np.random.randn(n_users, n_movies) * 0.5
mask = np.random.rand(n_users, n_movies) > 0.75
R_sparse = np.where(mask, R_noisy, 0)

# SVD completo sobre la matriz sparse
U, s, Vt = np.linalg.svd(R_sparse, full_matrices=False)

# Reconstrucción con k factores (similar a PCA con k componentes)
print("Reconstrucción con SVD truncado (Matriz de Ratings):")
print(f"  Valores singulares: {np.round(s[:10], 2)}")
print(f"  Varianza explicada acumulada:")

varianza_acum = np.cumsum(s**2) / np.sum(s**2)
for k in [1, 2, 3, 5, 10, 20]:
    R_k = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
    error = np.sqrt(np.mean((R - R_k)**2))
    print(f"    k={k:2d} factores: varianza={varianza_acum[k-1]:.1%}, RMSE={error:.4f}")

print(f"\n→ Con solo {n_factores_reales} factores (real) recuperamos la estructura latente")
print(f"  Memoria original: {R.nbytes} bytes")
print(f"  Memoria SVD-5:    {(U[:,:5].nbytes + s[:5].nbytes + Vt[:5,:].nbytes)} bytes")

---
## 🧠 Resumen de NumPy para ML

### Conceptos Clave
```
ndarray
  ├── dtype    → Elegir float32 para GPU, float64 para precisión numérica
  ├── strides  → Vista vs Copia: cuidado con modificaciones in-place
  ├── shape    → reshape/newaxis para broadcasting correcto
  └── flags    → Contiguidad afecta rendimiento de BLAS

Operaciones críticas en ML:
  @ (matmul)  → Forward/Backward pass, transformaciones lineales
  broadcasting → Normalización, pérdidas, activaciones
  fancy index  → Selección de batches, imputación condicional
  SVD          → PCA, recomendación, compresión, pseudoinversa
  lstsq        → Regresión lineal, solución de mínimos cuadrados
```

### ⚡ Reglas de Rendimiento:
1. **Nunca iterar sobre arrays** — usar operaciones vectorizadas
2. **float32 sobre float64** cuando la precisión lo permite (×2 velocidad en GPU)
3. **Preallocar arrays** con `np.zeros` en vez de hacer `append`
4. **Cuidado con copias**: `a = b` es alias, `a = b.copy()` es copia real